# Thought seeds: Art and prompts: the Met, Art Institute of Chicago, National Gallery of Art, Cleveland, Wikidata, Parti, DiffusionDB
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/thought-image-seeds/blob/main/colab/thought_seeds_art_prompts.ipynb)

Builds the seeds of these sources on this machine's CPU: downloads their annotations and metadata (never images), turns each picture
into a hidden picture specification in the dataset's attribute systems, checks every seed, and writes `seeds/<source>/` shards with
their statistics, plus LICENSES.md and CITATIONS.bib. The upload to
[AbstractPhil/thought-symbolic-image-captions](https://huggingface.co/datasets/AbstractPhil/thought-symbolic-image-captions) runs only when `UPLOAD = True` and an `HF_TOKEN` secret is set.
Run the cells in order; nothing needs a GPU.

In [ ]:
# C1: install the package (no dependency changes) and the pinned attribute systems
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "git+https://github.com/AbstractEyes/thought-image-seeds@main"], check=True)
if not os.path.isdir("/content/trainer_ref"):
    subprocess.run(["git", "clone", "-q", "https://github.com/AbstractEyes/geolip-nano-beatrix-trainer.git", "/content/trainer_ref"], check=True)
subprocess.run(["git", "-C", "/content/trainer_ref", "checkout", "-q", "8a7b2cb0236778ed8b6561128a246ebfd7c7ac67"], check=True)
os.environ["THOUGHT_TRAINER_REF"] = "/content/trainer_ref/geolip/nano_beatrix"
import thought_seeds
from thought_seeds import core
print("thought_seeds", thought_seeds.__version__, "| thought_pack", core.tp.VERSION, "|", len(core.KEYS), "keys")

In [ ]:
# C2: what this notebook builds
SOURCES = ['met', 'aic', 'nga', 'cleveland', 'wikidata', 'parti', 'diffusiondb']
LIMIT = None            # None = every item of each source; a number = a sample per source
RAW, OUT = "/content/raw", "/content/out"
UPLOAD = False          # True uploads OUT/seeds, LICENSES.md and CITATIONS.bib to the dataset repo
print(SOURCES, "| limit", LIMIT, "| upload", UPLOAD)

In [ ]:
# C3: fetch and build (prints the job size, the time spent and the time left per source)
from thought_seeds import build
results = build.run(SOURCES, RAW, OUT, limit=LIMIT, fetch=True)

In [ ]:
# C4: check every seed again from the files, and the statistics
import glob, json
from thought_seeds.core import tp, HELD
total, bad = 0, 0
for src in SOURCES:
    files = sorted(glob.glob(f"{OUT}/seeds/{src}/{src}-*.jsonl"))
    n = b = 0
    for f in files:
        for line in open(f, encoding="utf-8"):
            seed = json.loads(line)
            n += 1
            probs = list(tp.validate_seed(seed, HELD)) if hasattr(tp, "validate_seed") else []
            b += bool(probs)
    st = json.load(open(f"{OUT}/seeds/{src}/stats.json"))
    print(f"{src:22s} {n:8d} seeds  {st['held']:6d} held  {b:4d} failing  {len(files)} shards  hint {st['with_hint']}")
    total, bad = total + n, bad + b
print("all:", total, "seeds,", bad, "failing")
assert bad == 0, "a seed fails the gate: do not upload"

In [ ]:
# C5: the attribution files
build.docs(OUT)
print(open(f"{OUT}/LICENSES.md").read()[:3000])

In [ ]:
# C6: upload (only when UPLOAD is True and the HF_TOKEN secret exists)
if UPLOAD:
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    for src in SOURCES:
        api.upload_folder(folder_path=f"{OUT}/seeds/{src}", path_in_repo=f"seeds/{src}", repo_id="AbstractPhil/thought-symbolic-image-captions",
                          repo_type="dataset", commit_message=f"seeds: {src}")
        print("uploaded", src)
else:
    print("UPLOAD is False: nothing uploaded")